In [2]:
import sys
import os
import pandas as pd
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

print(project_root)  # para verificar

c:\Users\marco\OneDrive\Escritorio\Mis_proyectitos\electricity-price-forecast


In [3]:
from src.scraper import download_omie_price

In [4]:
from src.scraper import download_omie_range_v2

In [21]:
df = download_omie_range_v2('20230101','20231231')

20230101: 7 columnas
      0    1    2    3    4    5   6
0  2023  1.0  1.0  1.0  0.0  0.0 NaN
1  2023  1.0  1.0  2.0  0.0  0.0 NaN
OK: 20230101 (24 horas)
20230102: 7 columnas
      0    1    2    3      4      5   6
0  2023  1.0  2.0  1.0  61.38  61.38 NaN
1  2023  1.0  2.0  2.0  42.17  42.17 NaN
OK: 20230102 (24 horas)
20230103: 7 columnas
      0    1    2    3       4       5   6
0  2023  1.0  3.0  1.0  130.01  130.01 NaN
1  2023  1.0  3.0  2.0  120.00  120.00 NaN
OK: 20230103 (24 horas)
20230104: 7 columnas
      0    1    2    3       4       5   6
0  2023  1.0  4.0  1.0  119.99  119.99 NaN
1  2023  1.0  4.0  2.0  100.00  100.00 NaN
OK: 20230104 (24 horas)
20230105: 7 columnas
      0    1    2    3      4      5   6
0  2023  1.0  5.0  1.0  95.05  95.05 NaN
1  2023  1.0  5.0  2.0  79.21  79.21 NaN
OK: 20230105 (24 horas)
20230106: 7 columnas
      0    1    2    3       4       5   6
0  2023  1.0  6.0  1.0  107.21  107.21 NaN
1  2023  1.0  6.0  2.0  101.10  101.10 NaN
OK: 202301

In [22]:
df.shape

(8736, 7)

In [23]:
df.columns

Index(['year', 'month', 'day', 'hour', 'price_es', 'price_pt', 'datetime'], dtype='object')

# Let´s introduce SQLite

In [24]:
import sqlite3

# Crear base de datos SQLite
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
db_path = os.path.join(base_dir, "data", "electricity.db")
conn = sqlite3.connect(db_path)

# Guardar el dataframe en SQLite
df.to_sql("omie_prices", conn, if_exists="replace", index=False)
print(f"Tabla creada con {len(df)} filas")

# Verificar con una query SQL
result = pd.read_sql("SELECT COUNT(*) as total FROM omie_prices", conn)
print(result)

conn.close()

Tabla creada con 8736 filas
   total
0   8736


In [25]:
conn = sqlite3.connect(db_path)

# Precio medio por hora del día
pd.read_sql("""
    SELECT hour, AVG(price_es) as avg_price
    FROM omie_prices
    GROUP BY hour
    ORDER BY hour
""", conn)

,hour,avg_price
0,1.0,94.615934
1,2.0,87.648049
2,3.0,83.385165
3,4.0,81.098242
4,5.0,79.531264
5,6.0,82.528352
6,7.0,91.126016
7,8.0,101.341429
8,9.0,103.926374
9,10.0,93.321291


In [26]:
# Los 10 días con precio más alto
pd.read_sql("""
    SELECT year, month, day, AVG(price_es) as avg_price
    FROM omie_prices
    GROUP BY year, month, day
    ORDER BY avg_price DESC
    LIMIT 10
""", conn)

,year,month,day,avg_price
0,2023,2.0,21.0,151.427083
1,2023,2.0,24.0,148.078750
2,2023,3.0,6.0,147.770833
3,2023,2.0,13.0,146.767083
4,2023,3.0,1.0,145.911250
5,2023,2.0,16.0,145.327500
6,2023,2.0,15.0,145.070833
7,2023,2.0,9.0,144.996250
8,2023,2.0,22.0,144.449583
9,2023,3.0,5.0,143.924167


In [27]:
# Precio medio por mes
pd.read_sql("""
    SELECT month, AVG(price_es) as avg_price
    FROM omie_prices
    GROUP BY month
    ORDER BY month
""", conn)

#conn.close()

,month,avg_price
0,1.0,69.347554
1,2.0,134.230491
2,3.0,90.045989
3,4.0,76.959292
4,5.0,75.791208
5,6.0,95.586694
6,7.0,93.799879
7,8.0,97.859167
8,9.0,104.147694
9,10.0,89.744899
